<a href="https://colab.research.google.com/github/Song-yiJung/korean-modern-document-ocr/blob/main/01_step1_vision/step1_vision_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Step 1: Google Cloud Vision OCR**
*(매뉴얼 3.1장 대응)*

이 노트북은 사료 이미지에서 글자를 1차로 추출하는 도구이다. 이미지를 한 장씩 Google Cloud Vision API에 보내 텍스트를 뽑아내고, 폴더에 든 이미지를 한 번에 처리한다.

코랩(Colab)을 처음 쓴다면 다음 세 가지만 기억하면 된다.

* **위에서 아래로 셀을 차례대로 실행**한다. 각 셀 왼쪽의 ▷ 버튼을 누르거나 `Shift+Enter`를 누르면 실행된다.
* 본인이 **고쳐야 할 셀은 ③ 하나뿐**이다. 나머지는 내용을 바꾸지 말고 실행만 한다.
* 셀은 순서대로 실행해야 한다. 앞 셀을 건너뛰면 뒤 셀에서 오류가 난다.

**미리 준비할 것**

1. Vision API 키 파일(`.json`) — 발급 절차는 저장소의 환경 설정 문서(`docs/환경설정.md`) 참조. 발급하면 컴퓨터에 `.json` 파일이 내려받아지며, 이 파일을 본인 Google Drive에 올려 둔다.
2. 사료 이미지가 든 폴더 — Google Drive 안에 둔다.
3. 결과를 저장할 폴더 — Google Drive 안. 미리 만들지 않아도 노트북이 자동으로 만든다.

**예상 시간·비용**
* 처리 시간: 사료 한 장당 약 1~3초
* 비용: Vision 기준 월 1,000회까지 무료

## **시작 전: 이 노트북을 본인 계정으로 복사**

지금 열려 있는 노트북은 **원본이라 읽기 전용**이다. 그대로는 실행하거나 수정 내용을 저장할 수 없으므로, 먼저 본인 계정으로 복사한다.

상단 메뉴에서 **`파일 → Drive에 사본 저장`**을 누른다. 사본이 본인 Google Drive의 `Colab Notebooks` 폴더에 저장되고 새 탭으로 열린다. **이후 작업은 모두 이 사본에서 한다.** 다음에 다시 작업할 때도 이 사본을 열면 설정이 그대로 남아 있다.

#### ① 필요한 패키지 설치

Vision API를 쓰는 데 필요한 프로그램을 설치하는 셀이다. 코랩은 일정 시간 쓰지 않거나 창을 닫으면 환경이 초기화되어 설치한 것이 사라진다. 그래서 **새로 시작할 때마다 이 셀을 가장 먼저 실행**한다(올려 둔 파일은 Drive에 있으므로 사라지지 않는다).

`!` 기호는 "설치 명령으로 실행하라"는 표시이다. 의미만 알아 두면 되고 고칠 필요는 없다.

In [ ]:
!pip install -q google-cloud-vision

### ② Google Drive 연결(마운트)

사료 이미지를 읽고 결과를 저장하려면 코랩이 본인 Drive에 접근할 수 있어야 한다. 이 셀이 그 연결(마운트)을 한다.

실행하면 권한 요청 창이 표시된다. 본인 구글 계정을 선택하고 안내에 따라 **허용**을 누른다. `Mounted at /content/drive`가 출력되면 정상이다. 이후 `/content/drive/MyDrive/` 경로로 본인 Drive의 최상위(내 드라이브)에 접근할 수 있다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### ③ ★ 사용자 설정 — *여기만* 본인 환경에 맞게 수정

**본 노트북에서 직접 고치는 곳은 아래 코드 셀 하나뿐이다.** 네 값을 본인 환경에 맞게 바꾼 뒤 실행한다.

---

**[1] `VISION_KEY_SECRET_NAME` — 키 파일 경로를 담아 둘 "보안 비밀"의 이름**

키 파일 경로를 코드에 그대로 적으면 노트북을 공유할 때 키가 노출될 수 있다. 코랩에는 이런 정보를 코드와 분리해 보관하는 **보안 비밀(Secret)** 기능이 있다. 여기에는 키 파일의 *내용*이 아니라 *파일이 있는 경로*를 저장한다.

처음이라면 다음 순서를 그대로 따른다.

1. 발급받은 `.json` 키 파일을 본인 Google Drive에 올린다(예: `내 드라이브/keys/`).
2. 그 파일의 코랩 경로를 확인한다. 보통 `/content/drive/MyDrive/keys/vision_key.json` 형태이다(`내 드라이브`가 코랩에서는 `MyDrive`로 표시된다).
3. 코랩 화면 **왼쪽 사이드바의 🔑(열쇠) 아이콘**을 누른다.
4. **새 보안 비밀**을 추가한다.
5. **이름** 칸에 `VISION_KEY_PATH`를 입력한다. *(아래 코드의 변수 값과 글자 그대로 같아야 한다.)*
6. **값** 칸에 2번에서 확인한 경로를 붙여 넣는다.
7. **노트북 액세스** 토글을 켠다.

> 키 파일은 Drive에 한 부만 두고, 보안 비밀에는 "그 파일이 어디 있는지"만 적는 것이다. 이 방식이면 다음 단계인 step2 노트북에서도 같은 보안 비밀을 그대로 쓸 수 있다.
>
> 참고: step1(Vision) 키는 *파일*이라 경로를 저장하지만, step2(Gemini) 키는 `AIza…`로 시작하는 *문자열*이라 그때는 경로가 아니라 키 문자열 자체를 값에 넣는다.

---

**[2] `INPUT_DIR`** — 사료 이미지가 든 Drive 폴더 경로. 하위 폴더의 이미지까지 자동으로 모으므로, 권·문건·연도별로 폴더를 나눠 두어도 그대로 작동한다.

**[3] `OUTPUT_DIR`** — 결과를 저장할 폴더 경로. 없으면 자동으로 만든다.

**[4] `LANGUAGE_HINTS`** — 사료의 주요 언어 힌트(2~3개 권장). 일본어·구자체 한자 → `['ja','zh-Hant']`, 한글 → `['ko','en']`, 한문 → `['zh-Hant','zh-Hans']`, 영문 → `['en']`. 4개 이상은 오히려 정확도가 떨어질 수 있다.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ③ ★ 사용자 설정 — 아래 4개 값만 본인 환경에 맞게 수정
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# [1] Vision API 키 경로를 등록한 Colab Secret의 이름 (위 설명 참조)
VISION_KEY_SECRET_NAME = 'VISION_KEY_PATH'

# [2] 사료 이미지가 있는 Drive 폴더 (하위 폴더까지 자동 수집)
INPUT_DIR = '/content/drive/MyDrive/사료_이미지'

# [3] 결과 저장 폴더 (없으면 자동 생성)
OUTPUT_DIR = '/content/drive/MyDrive/vision_결과'

# [4] 사료의 주요 언어 힌트 (2~3개 권장)
LANGUAGE_HINTS = ['ja', 'zh-Hant']


### ③-보충 (선택) 샘플 이미지로 먼저 시험하기

본인 사료를 준비하기 전에, 저장소가 제공하는 공개 샘플 이미지로 한 번 돌려 볼 수 있다. 이 셀을 실행하면 저장소의 `samples/` 폴더 이미지가 위 ③에서 정한 `INPUT_DIR`로 복사된다(③을 먼저 실행해 `INPUT_DIR`이 정해진 뒤라야 한다).

**본인 사료로 작업할 때는 이 셀을 실행하지 말고**, 본인 이미지를 `INPUT_DIR` 폴더에 직접 올린다.

In [ ]:
# (선택) 첫 실습용 샘플 이미지를 저장소에서 INPUT_DIR로 내려받기 (하위 폴더 구조 유지)
# 본인 사료로 작업할 때는 이 셀을 건너뛴다.
import shutil
from pathlib import Path

shutil.rmtree('/content/_repo', ignore_errors=True)
!git clone --depth 1 https://github.com/Song-yiJung/korean-modern-document-ocr.git /content/_repo

src = Path('/content/_repo/samples')
dst = Path(INPUT_DIR); dst.mkdir(parents=True, exist_ok=True)
exts = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
n = 0
for p in src.rglob('*'):
    if p.is_file() and p.suffix.lower() in exts:
        target = dst / p.relative_to(src)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(p, target)
        n += 1
print(f"샘플 {n}장을 {dst} 에 (하위 폴더 구조 포함) 복사했습니다.")


### ④ Vision API 인증 + 폴더 점검

③에서 등록한 보안 비밀을 읽어 Vision에 로그인하고, 입력·출력 폴더가 제대로 지정됐는지 점검하는 셀이다.

`✔ Vision API 인증 정보 등록 완료`와 함께 입력·출력 폴더 경로가 출력되면 정상이다. 오류가 나면 화면의 안내 문구를 읽고 다음을 확인한다 — 보안 비밀이 등록됐는지, 이름이 `VISION_KEY_PATH`와 정확히 같은지, "노트북 액세스" 토글이 켜져 있는지, ③의 `INPUT_DIR` 폴더가 Drive에 실제로 있는지.

In [ ]:
import os
from pathlib import Path
from google.colab import userdata

# 1) Secret에서 키 파일 경로 읽기
try:
    key_path = userdata.get(VISION_KEY_SECRET_NAME)
except Exception as e:
    raise RuntimeError(
        f"Colab Secret '{VISION_KEY_SECRET_NAME}' 접근 실패: {e}\n"
        "→ 좌측 🔑 아이콘에서 (1) 시크릿이 등록되어 있는지, "
        "(2) 이름이 정확한지, (3) '노트북 액세스' 토글이 켜져 있는지 확인하세요."
    )

# 2) 환경변수 등록
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = key_path
print(f"✔ Vision API 인증 정보 등록 완료")
print(f"  키 파일: {key_path}")

# 3) 입출력 폴더 검증
INPUT_DIR  = Path(INPUT_DIR)
OUTPUT_DIR = Path(OUTPUT_DIR)

if not INPUT_DIR.exists():
    raise RuntimeError(
        f"입력 폴더가 존재하지 않습니다: {INPUT_DIR}\n"
        "→ 셀 ③의 INPUT_DIR 경로가 맞는지, "
        "Drive에 그 이름의 폴더를 실제로 만들었는지 확인하세요."
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"  입력 폴더: {INPUT_DIR}")
print(f"  출력 폴더: {OUTPUT_DIR}")

### ⑤ Vision 호출 함수 정의

이미지 한 장을 받아 Vision으로 처리하는 함수를 만들어 두는 셀이다. 고칠 것은 없고 실행만 하면 된다. 이 함수는 두 가지를 돌려준다.

* `text` — 본문 텍스트(검색과 다음 교정 단계의 입력용)
* `full` — 글자별 좌표·신뢰도 등 구조 정보(지금은 저장만 해 두고, 나중에 디지털 판본을 만들 때 활용)

사료처럼 글자가 빽빽하고 행·단락 구조가 분명한 문서에는 `document_text_detection` 방식이 적합해 이 함수가 그것을 사용한다.

In [ ]:
from google.cloud import vision
from google.protobuf import json_format

vision_client = vision.ImageAnnotatorClient()


def extract_text_from_image(img_path: Path):
    """이미지 한 장을 Vision API로 처리 → (텍스트, 구조 dict) 반환."""
    with open(img_path, "rb") as f:
        content = f.read()

    image = vision.Image(content=content)
    image_context = vision.ImageContext(language_hints=LANGUAGE_HINTS)

    response = vision_client.document_text_detection(
        image=image,
        image_context=image_context,
    )

    if response.error.message:
        raise Exception(f"Vision API 통신 오류: {response.error.message}")

    if response.full_text_annotation:
        text = response.full_text_annotation.text.strip()
        full = json_format.MessageToDict(response.full_text_annotation._pb)
        return text, full
    return "", {}


print("✔ Vision 호출 함수 정의 완료")

### ⑥ 처리 대상 이미지 모으기

`INPUT_DIR`과 그 하위 폴더에서 이미지(`.jpg`/`.jpeg`/`.png`/`.tif`/`.tiff`)를 모두 모은다. 실행하면 처리 대상이 몇 장인지, 처음 세 장이 무엇인지 출력된다. **이 건수가 예상과 맞는지 먼저 확인**한다.

`이미지가 한 장도 없습니다`라고 나오면 ③의 `INPUT_DIR` 경로가 맞는지, 이미지를 실제로 그 폴더에 올렸는지 확인한다.

In [ ]:
valid_extensions = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}
image_paths = sorted([
    p for p in INPUT_DIR.rglob('*')
    if p.suffix.lower() in valid_extensions
])

print(f"처리 대상: 총 {len(image_paths)}장")
if image_paths:
    print("처음 3장:")
    for p in image_paths[:3]:
        print(f"  {p.relative_to(INPUT_DIR)}")
else:
    print("⚠️ 이미지가 한 장도 없습니다.")
    print("   → INPUT_DIR 경로와 업로드 여부를 확인하세요.")

### ⑦ 일괄 처리 (실제 OCR 실행)

모은 이미지를 한 장씩 Vision에 보내 결과를 저장하는, 이 노트북의 핵심 셀이다. 장수가 많으면 시간이 걸리며 진행 상황이 한 줄씩 출력된다. 두 가지 안전장치가 들어 있다.

* **중복 처리 방지**: 이미 결과가 있는 사료는 건너뛴다. 처리가 중간에 끊겨도 다시 실행하면 남은 분량만 이어서 처리한다(100장 중 60장까지 하고 멈췄다면 남은 40장만 처리).
* **오류 분리 기록**: 한 장에서 일시 오류(이미지 손상, 네트워크 끊김 등)가 나도 전체가 멈추지 않는다. 그 장만 `.err.json`으로 따로 기록되고, 다음에 이 셀을 다시 실행하면 자동 재시도된다.

**출력 폴더 구조**
```
결과 폴더/
  ├─ <파일ID>.json       ← 본문 텍스트 + 좌표 구조 + 메타데이터
  ├─ <파일ID>.txt        ← 본문 텍스트만 (사람이 읽기 쉬움)
  └─ <파일ID>.err.json   ← (오류 시) 다음 실행 때 자동 재시도
```
`.json`은 step2의 입력으로, `.txt`는 결과를 눈으로 빠르게 훑어볼 때 쓴다.

In [ ]:
import json
from datetime import datetime

processed = skipped = errors = 0

for idx, img_path in enumerate(image_paths, 1):
    rel_path = img_path.relative_to(INPUT_DIR)
    out_folder = OUTPUT_DIR / rel_path.parent
    out_folder.mkdir(parents=True, exist_ok=True)

    base = img_path.stem
    json_path = out_folder / f"{base}.json"
    err_path  = out_folder / f"{base}.err.json"
    txt_path  = out_folder / f"{base}.txt"

    # 멱등성: 정상 JSON이 이미 있으면 스킵
    if json_path.exists():
        skipped += 1
        continue

    print(f"[{idx}/{len(image_paths)}] {rel_path}", flush=True)
    try:
        text, full = extract_text_from_image(img_path)

        result = {
            "file_path": str(rel_path),
            "vision_raw": text,
            "vision_full": full,            # 좌표·신뢰도 보존 (향후 활용)
            "processed_at": datetime.now().isoformat(timespec='seconds'),
            "gemini_corrected": ""          # step2가 채울 자리
        }
        json_path.write_text(
            json.dumps(result, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        # 과거 에러 기록 정리 (이번에 성공했으므로)
        if err_path.exists():
            err_path.unlink()

        if text:
            txt_path.write_text(text, encoding="utf-8")
            print(f"   -> {len(text)}자 추출")
        else:
            print(f"   -> 텍스트 없음 (도면·사진·공백 페이지 가능)")
        processed += 1

    except Exception as e:
        err = {
            "file_path": str(rel_path),
            "error": str(e),
            "failed_at": datetime.now().isoformat(timespec='seconds')
        }
        err_path.write_text(
            json.dumps(err, ensure_ascii=False, indent=2),
            encoding="utf-8"
        )
        print(f"   -> 오류: {str(e)[:100]}")
        errors += 1

print(f"\n완료. 처리 {processed} / 스킵 {skipped} / 오류 {errors}")


### ⑧ 결과 점검

처리가 잘 됐는지 자동으로 점검하는 셀이다. 세 가지를 출력한다.

* 입력한 이미지 수와 결과 파일 수가 맞는지
* 글자가 50자 미만인 페이지 목록 (이미지 품질 문제의 신호일 수 있다. 다만 도면·사진·빈 페이지는 원래 글자가 적으므로 무조건 불량으로 보지 말고, 살펴볼 페이지를 가려내는 신호로만 쓴다)
* 오류 기록 건수 (있으면 ⑦을 다시 실행해 재시도)

결과가 비었거나 글자가 너무 적은 페이지가 있으면 원본 이미지의 해상도·밝기·방향을 확인한다. 스캔 단계에서 생긴 품질 문제는 OCR로 바로잡기 어려우므로, 원본을 다시 구하거나 그 페이지만 따로 처리한다.

In [ ]:
all_jsons = list(OUTPUT_DIR.rglob('*.json'))
ok_jsons  = [f for f in all_jsons if not f.name.endswith('.err.json')]
err_jsons = [f for f in all_jsons if f.name.endswith('.err.json')]

print(f"정상 결과: {len(ok_jsons)} / 입력: {len(image_paths)}")
print(f"에러 기록: {len(err_jsons)}건  (있으면 셀 ⑦ 다시 실행하면 자동 재시도)")

print("\n--- 글자 50자 미만 페이지 점검 ---")
short_ones = []
for jp in ok_jsons:
    data = json.loads(jp.read_text(encoding='utf-8'))
    n = len(data.get('vision_raw', ''))
    if n < 50:
        short_ones.append((jp.stem, n))

if not short_ones:
    print("  없음.")
else:
    for fid, n in short_ones:
        print(f"  {fid}: {n}자")
print(f"\n총 {len(short_ones)}건")

### ⑨ 다음 단계

여기까지 마치면 사료 한 장당 JSON 하나가 `OUTPUT_DIR`에 생성된다. 각 JSON에는 다음이 담긴다.

* `vision_raw` — 1차 추출 본문 텍스트 (step2의 입력)
* `vision_full` — 글자별 좌표·신뢰도 (나중에 디지털 판본 단계에서 활용)
* `gemini_corrected` — 아직 빈 값 (step2가 채움)

이 결과는 아직 완성본이 아니다. 글자가 추출됐을 뿐 행 순서가 흐트러져 있고, 모양이 비슷한 글자의 오인식이나 없는 글자를 만들어 내는 오류가 남아 있다. 이를 문맥에 맞게 교정하는 작업은 다음 노트북 `step2_gemini_colab.ipynb`에서 이어진다.